# Phase 3: Fuzzy Systems

`FS_Model.ipynb`

---

Team 1 -- NYC Taxi Trip Duration Prediction

CSE 5140 | Spring 2026 | CSUSB


## Glossary -- Key Terms Used in This Notebook

| Term | Definition |
|:-----|:-----------|
| **Fuzzy Set** | A set where membership is not binary (0 or 1) but a degree between 0 and 1. For example, a 12 km trip might be 0.7 "medium" and 0.3 "long." |
| **Membership Function** | A curve that maps a crisp input value to a membership degree (0-1). Common shapes: triangular, trapezoidal, Gaussian. |
| **Linguistic Variable** | A variable described with words instead of numbers -- e.g., trip distance is "short," "medium," or "long." |
| **Fuzzy Rule** | An IF-THEN statement using linguistic variables -- e.g., *IF distance IS long AND hour IS rush_hour THEN duration IS very_long.* |
| **Fuzzy Inference System** | The complete system: fuzzify inputs -> apply rules -> defuzzify to get a crisp predicted value. |
| **Defuzzification** | Converting fuzzy output back into a single number (e.g., centroid method). |
| **scikit-fuzzy** | Python library (`skfuzzy`) for building fuzzy inference systems. |
| **Interpretability** | How easy it is for a human to understand *why* the model made a prediction. Measured by rule count, rule length, and feature count. |
| **H3** | Phase 3 hypothesis: Does the fuzzy system provide meaningful interpretability while maintaining acceptable predictive performance compared to the NN (Phase 1) and EA (Phase 2)? |


## What This Notebook Does

In this notebook, we build a **Fuzzy Inference System (FIS)** to predict NYC taxi trip duration.
Unlike the neural network (Phase 1) and the evolutionary algorithm (Phase 2), the fuzzy system is designed for **interpretability** --
the goal is a model whose reasoning a human can read and understand, expressed as simple IF-THEN rules using everyday language like "short distance" or "rush hour."

> **Design philosophy:** We intentionally trade some prediction accuracy for transparency. The fuzzy system will not beat the NN on raw metrics, and that is expected and acceptable. The value is in producing a model where every prediction can be explained.

---

## Phases at a Glance

| Phase | Method | Notebook | Primary Goal |
|:-----:|:-------|:---------|:-------------|
| 1 | Neural Network | `NN_Analysis.ipynb` | Maximize predictive accuracy |
| 2 | Evolutionary Algorithm | `EA_Optimization.ipynb` | Optimize NN hyperparameters via GA |
| **3** | **Fuzzy System** | **`FS_Model.ipynb`** | **Interpretable prediction via fuzzy rules** |


## How to Run This Notebook

| Step | Instruction |
|:----:|:------------|
| 1 | **Open in NRP JupyterHub** (required for final results). You may test locally, but all final outputs must be executed on NRP using the project's standard environment. |
| 2 | **Run cells in order from top to bottom.** Each cell depends on earlier cells. |
| 3 | **Do not modify constants** (seed, split sizes, data path). These must remain consistent with Phases 1 and 2 to ensure fair comparison. |

> **Execution environment:** The fuzzy system is implemented in Python using `scikit-fuzzy` (`skfuzzy`). The notebook runs within the project's standard TensorFlow-based execution environment, ensuring consistency across all phases. The framework requirement refers to the *execution environment*, not the modeling approach. The fuzzy system remains a **true fuzzy inference system**, preserving interpretability.


## Fair Evaluation Rules

| Rule | Detail |
|:-----|:-------|
| **Same dataset** | First 1,000,000 rows from `train.csv` |
| **Same split** | 70% train / 15% validation / 15% test (identical to Phases 1 and 2) |
| **Same seed** | All random operations use the same fixed seed as prior phases |
| **No re-shuffling** | Data order is identical to Phases 1 and 2 |
| **Test set untouched** | The 15% test set is used **exactly once** for final evaluation in Step 3 |
| **Validation only for design** | All membership function tuning and rule refinement uses the validation set |
| **No target leakage** | No feature depends on future or target information |

> **These rules ensure:** Fair comparison across NN, EA, and FS | No artificial performance inflation | Strong reproducibility and methodological rigor

---

## Common Mistakes to Avoid

| Mistake | Why It Matters |
|:--------|:---------------|
| **Using test set during design** | Checking test performance while tuning membership functions or rules |
| **Accidentally reshuffling data** | Re-running shuffle operations breaks consistency with earlier phases |
| **Changing split ratios** | Using a different train/validation/test split than Phases 1 and 2 |
| **Feature leakage** | Creating features that use future information or indirectly encode the target |
| **Tuning on the test set** | Adjusting rules or parameters after seeing test results |
| **Overcomplicating the FS** | Adding too many features or memberships, making the system less interpretable |

> **Warning:** If any of these occur, the results become invalid for fair comparison and may significantly impact grading.


# Step 1: Fuzzy System Design & Setup

Define the objective, select features, design membership functions, and lock the methodology.

---

## 1.1 -- Fuzzy System Objective

| Item | Detail |
|:-----|:-------|
| **Prediction target** | `trip_duration` (seconds), consistent with Phases 1 and 2 |
| **Purpose** | Build a fuzzy inference system that provides **human-readable, rule-based predictions** for NYC taxi trip duration. Every prediction can be traced back to IF-THEN rules using linguistically meaningful terms. |
| **Hypothesis (H3)** | The fuzzy system provides meaningful interpretability (measured by number of rules, average rule length, and feature coverage) while maintaining acceptable predictive performance compared to the Phase 1 NN and Phase 2 EA-optimized NN. |
| **Execution** | Final execution on **NRP JupyterHub**. TensorFlow may be used only for efficient batched computation; the model itself remains a **true fuzzy inference system** with explicitly defined variables, membership functions, and rule structure. |
| **Acceptable accuracy** | We do *not* require the fuzzy system to match the NN's R-squared. A meaningful drop is justified if the model is fully transparent -- a non-expert can read the rules and understand *why* a trip was predicted to be long or short. |

> **Data & Fairness Protocol:** Same first 1,000,000 rows | Reused exact train/val/test split from Phase 1/2 | No reshuffling | Validation for design/tuning only | Test set untouched until final evaluation

| Steps 1-2 | Step 3 | Goal |
|:----------:|:------:|:----:|
| Validation set only | One-time test evaluation | Interpretability + performance |


## 1.2 -- Feature Selection: Evidence from Phases 1 and 2

> **Why we select carefully:** Fuzzy systems scale poorly with many inputs. Each input feature multiplies the potential rule space. The project FAQ explicitly warns about this. We target **6 input features**.

### Phase 1 NN Permutation Importance (Validation Set, R-squared log drop)

| Rank | Feature | R-squared log Drop | Std | Family |
|:----:|:--------|--------------------:|----:|:-------|
| **1** | `haversine_km` | **2.0482** | 0.0064 | Spatial |
| 2 | `hour_cos` | 0.3014 | 0.0006 | Temporal |
| 3 | `delta_lat` | 0.2860 | 0.0011 | Spatial |
| 4 | `pickup_hour` | 0.1077 | 0.0007 | Temporal |
| 5 | `dow_cos` | 0.0846 | 0.0004 | Temporal |
| 6 | `delta_lon` | 0.0644 | 0.0010 | Spatial |
| 7 | `pickup_dow` | 0.0477 | 0.0003 | Temporal |
| 8 | `passenger_count` | 0.0038 | 0.0002 | Context |

### Grouped Importance by Family

| Family | Total R-squared log Drop | Interpretation |
|:-------|-------------------------:|:---------------|
| **Spatial** | **2.399** | Dominant -- distance and direction are by far the strongest predictors |
| Temporal | 0.639 | Strong -- time of day and day of week matter significantly |
| Context | 0.123 | Weak -- vendor and passenger info contribute minimally |

> **Phase 2 EA finding:** The EA-optimized NN did *not* improve over the Phase 1 NN (delta R-squared = -0.026, bootstrap CI includes 0). This confirms the Phase 1 feature set was already near-optimal and Phase 3 should focus on **interpretability rather than accuracy improvements**.

> **Design direction:** Use a small subset of high-impact features (primarily spatial + temporal) to build an interpretable fuzzy system.


## 1.3 -- Selected Input Features (6 Features)

| # | Feature | Family | NN Rank | R-squared Drop | Why Selected | Expected Influence |
|:-:|:--------|:-------|:-------:|---------------:|:-------------|:-------------------|
| **1** | `haversine_km` | Spatial | 1 | **2.048** | By far the most important feature. Directly interpretable as "how far." | Longer distance -> longer trip |
| 2 | `pickup_hour` | Temporal | 4 | 0.108 | Maps naturally to fuzzy labels: "early morning," "rush hour," "night." | Rush hours -> longer; night -> shorter |
| 3 | `pickup_dow` | Temporal | 7 | 0.048 | Day of week maps to "weekday" vs "weekend" patterns. | Weekdays -> more traffic |
| 4 | `delta_lat` | Spatial | 3 | 0.286 | North-south displacement captures directional travel patterns. | Large displacement -> longer trip |
| 5 | `delta_lon` | Spatial | 6 | 0.064 | East-west displacement; complements delta_lat for direction. | Large displacement -> longer trip |
| 6 | `passenger_count` | Context | 8 | 0.004 | Weakest, but the only interpretable context variable. | Minimal; may correlate with trip type |

### Features NOT Selected

| Feature | Why Excluded |
|:--------|:-------------|
| **`hour_sin`, `hour_cos`** | Cyclic encodings are useful for NNs but not linguistically interpretable. We use raw `pickup_hour` instead. |
| **`dow_sin`, `dow_cos`** | Same reasoning -- raw `pickup_dow` is more natural for fuzzy rules. |
| **`pickup_month`** | Low importance. Adds complexity without proportional interpretability gain. |
| **`vendor_1`, `vendor_2`** | Binary categorical. No ordinal or continuous structure for fuzzy membership. |
| **`store_and_fwd_Y`** | Negligible importance (NN drop = 0.0001). Nearly zero variance. |


## 1.4 -- Output Variable Design

**Output variable:** `predicted_duration` (seconds). Uses **5 fuzzy labels** spanning typical NYC taxi trip durations.
Boundaries are informed by the observed training-data distribution (Phase 1), where most trips fall between 3-30 minutes with a right-skewed tail.

| Label | Range (sec) | Range (min) | Description |
|:------|------------:|------------:|:------------|
| `very_short` | 0 - 300 | 0 - 5 | Quick trips: same-neighborhood, short hops |
| `short` | 120 - 900 | 2 - 15 | Typical short urban trips |
| `medium` | 600 - 1800 | 10 - 30 | Average-length trips, moderate distance |
| `long` | 1200 - 3600 | 20 - 60 | Cross-borough or congested trips |
| `very_long` | 2400 - 5400+ | 40 - 90+ | Airport runs, outer-borough, or heavy traffic |

| Overlap | Shapes | Defuzzification |
|:--------|:-------|:----------------|
| Ranges deliberately overlap. A 15-min trip might be 0.4 "short" and 0.6 "medium." | Triangular middle sets + trapezoidal edge sets for simple, interpretable transitions. | Centroid (center of gravity) -- the most common and interpretable approach. |


## 1.5 -- Membership Parameter Table (Implementation-Ready Spec)

Converts the approved fuzzy variables into a **numerically disciplined design**.
These are the **locked implementation-ready values** for the first-pass fuzzy system.

| Variable | Family | Universe | # Sets | Labels | Shape | Exact Parameters | Role |
|:---------|:-------|:---------|:------:|:-------|:------|:-----------------|:-----|
| `haversine_km` | Spatial | 0-30 km | 3 | short, medium, long | trap/tri/trap | `[0,0,1.5,4]` \| `[2.5,7.5,12.5]` \| `[10,16,30,30]` | Core baseline driver |
| `pickup_hour` | Temporal | 0-23 h | 5 | early_morning, morning_rush, midday, evening_rush, night | trap/tri/tri/tri/trap | `[0,0,5,7]` \| `[6,8,10]` \| `[10,13,16]` \| `[15,18,20]` \| `[19,21,23,23]` | Traffic-pattern modifier |
| `pickup_dow` | Temporal | 0-6 | 2 | weekday, weekend | trap/trap | `[0,0,3.5,4.5]` \| `[4.5,5,6,6]` | Day-type modifier |
| `delta_lat` | Spatial | -0.20 to 0.20 deg | 3 | south, neutral, north | tri/tri/tri | `[-0.20,-0.10,-0.01]` \| `[-0.03,0,0.03]` \| `[0.01,0.10,0.20]` | N/S direction modifier |
| `delta_lon` | Spatial | -0.20 to 0.20 deg | 3 | west, neutral, east | tri/tri/tri | `[-0.20,-0.10,-0.01]` \| `[-0.03,0,0.03]` \| `[0.01,0.10,0.20]` | E/W direction modifier |
| `passenger_count` | Context | 1-6 pax | 3 | low, medium, high | trap/tri/trap | `[1,1,1.5,2]` \| `[1.5,3,4.5]` \| `[4,5,6,6]` | Sparse tie-breaker |
| `predicted_duration` | Output | 0-5400 s | 5 | very_short, short, medium, long, very_long | trap/tri/tri/tri/trap | `[0,0,180,420]` \| `[240,600,960]` \| `[780,1500,2280]` \| `[1800,3000,4200]` \| `[3600,4500,5400,5400]` | Consequent space |

| Trapezoidal edges | Triangular middles | Overlap is intentional |
|:------------------|:-------------------|:-----------------------|
| Avoid awkward behavior at min/max boundaries. | Keep interpretation simple and plots easy to explain. | A trip can be partly "short" and partly "medium" simultaneously. |


## 1.6 -- Interpretability Budget and Complexity Caps

| Metric | Locked Target | Rationale |
|:-------|:-------------:|:----------|
| **Input features** | **6** | Keeps model within team target; avoids rule-space explosion. |
| **Memberships per input** | **Mostly 3** | Three labels suffice for low/medium/high reasoning. Exceptions: `pickup_hour` (5), `pickup_dow` (2). |
| **Theoretical full rule space** | **810** | 3 x 5 x 2 x 3 x 3 x 3 = 810. Far too large to use directly. |
| **Target active rules** | **25 - 40** | Large enough for coverage, small enough to explain in a report. |
| **Hard rule ceiling** | **50** | Above this weakens interpretability. |
| **Target avg rule length** | **2 - 3 antecedents** | Shorter rules are easier to justify and read. |
| **Hard antecedent ceiling** | **4** | Longer rules become hard to explain and maintain. |
| **Feature coverage** | **100%** | Every input appears in at least 1 meaningful rule. |
| **Redundancy** | **None** | No near-duplicate rules allowed. |

> **Complexity-control strategy:** Avoid exhaustive rule generation. Build a compact expert rule base around the strongest interactions: `haversine_km` + `pickup_hour`, `haversine_km` + `pickup_dow`, `haversine_km` + directional displacement. Start with 2-antecedent rules; add 3-antecedent exceptions only when they clearly improve interpretability.


## 1.7 -- Initial Rule Template Set

| Rule Family | Antecedent Pattern | Expected Consequent | Rationale |
|:------------|:-------------------|:--------------------|:----------|
| **Distance baseline** | `haversine_km` only | very_short / medium / long | Simplest, most interpretable backbone. |
| **Distance + rush-hour** | `haversine_km` + `pickup_hour` (rush) | Shift -> long / very_long | Strongest traffic-driven delay pattern. |
| **Distance + night relief** | `haversine_km` + `pickup_hour` (night) | Shift -> very_short / short | Lower-traffic nighttime pattern. |
| **Distance + day type** | `haversine_km` + `pickup_dow` | Longer weekday, simpler weekend | Interpretable weekday/weekend effects. |
| **Distance + direction** | `haversine_km` + `delta_lat` / `delta_lon` | medium / long / very_long | Readable spatial-direction adjustment. |
| **Context exceptions** | `haversine_km` + `passenger_count` | Mild upward shift for high pax | Sparse context modifier. |
| **3-antecedent exceptions** | `haversine_km` + timing + day/direction | long / very_long for busiest patterns | Small set of richer rules for high-value edges. |

> **Encoding order:** 1) Distance-only baselines -> 2) Hour-based traffic modifiers -> 3) Weekday/weekend adjustments -> 4) Directional modifiers -> 5) Small number of passenger-count or 3-antecedent exceptions.


## 1.8 -- Methodology Lock

| Constraint | Locked Value | Reason |
|:-----------|:-------------|:-------|
| **Dataset** | First 1,000,000 rows of `train.csv` | Same as Phases 1 and 2 |
| **Split** | 70/15/15 train/val/test, same seed | Reuse exact Phase 1 split; no re-shuffling |
| **Test set** | Untouched until Step 3 | Prevents overfitting to test data |
| **Validation use** | Design, tuning, rule refinement only | All Steps 1-2 decisions validated here |
| **Implementation** | Python + scikit-fuzzy (`skfuzzy`) | Must be a real FIS, not a neural surrogate |
| **Execution env** | NRP JupyterHub; TF optional for batched compute | Compliant while preserving true FIS |
| **Random seed** | 42 (same as Phases 1 and 2) | Reproducibility |
| **Defuzzification** | Centroid method | Most common, most interpretable |

> **Compliance statement:** This Phase 3 implementation reuses the controlled data split from Phases 1 and 2 without re-shuffling. The test set remains untouched until one-time final evaluation in Step 3. The fuzzy system is a genuine FIS using scikit-fuzzy, running within the NRP JupyterHub environment. No features introduce target leakage or depend on future information.

> **Critical:** Violating any of these constraints invalidates the experimental comparison and compromises project fairness.


## Step 1 Closeout Summary

| Deliverable | Status | Location |
|:------------|:------:|:---------|
| FS objective statement | **Complete** | Section 1.1 |
| Feature selection with evidence | **Complete** | Sections 1.2-1.3 |
| Output variable design | **Complete** | Section 1.4 |
| Membership parameter table | **Complete** | Section 1.5 |
| Interpretability budget | **Complete** | Section 1.6 |
| Initial rule template set | **Complete** | Section 1.7 |
| Methodology lock and compliance | **Complete** | Section 1.8 |


# Step 2: Fuzzy System Implementation

Code cells below are aligned with the locked Step 1 design. Step 1 closeout is documented above.


> **Cell 1 -- Imports, Constants & Environment**
>
> Load all required libraries (`numpy`, `pandas`, `skfuzzy`, `sklearn`) and define constants matching Phases 1 and 2.


In [ ]:

# ============================================================
# Cell 1 — Imports, Constants & Environment
# ============================================================
# Must match Phase 1/2: SEED, NROWS, DATA_PATH, split ratios.

import os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

# --- Install scikit-fuzzy if needed ---
%pip install -q scikit-fuzzy
import skfuzzy as fuzz
import skfuzzy.control as ctrl

# ---- Constants (identical to Phase 1 / Phase 2) ----
SEED       = 42
NROWS      = 1_000_000
DATA_PATH  = Path("../data/train.csv")
TARGET     = "trip_duration"

# Split ratios (overall 70 / 15 / 15)
TEST_SIZE  = 0.15          # holdout from full data
VAL_RATIO  = 0.15 / 0.85   # validation from dev set

# The 6 fuzzy input features selected in Step 1
FUZZY_FEATURES = [
    "haversine_km",
    "pickup_hour",
    "pickup_dow",
    "delta_lat",
    "delta_lon",
    "passenger_count",
]

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

print(f"SEED       = {SEED}")
print(f"NROWS      = {NROWS:,}")
print(f"DATA_PATH  = {DATA_PATH}")
print(f"TARGET     = {TARGET}")
print(f"Features   = {FUZZY_FEATURES}")
print(f"skfuzzy    = {fuzz.__version__}")


> **Cell 2 -- Data Load & Split Verification**
>
> Load the same 1M rows, apply the same shuffle and split, verify row counts match Phases 1 and 2.


In [ ]:

# ============================================================
# Cell 2 — Data Load & Split Verification
# Reuses exact loading and splitting from Phase 1.
# ============================================================

import urllib.request

# Ensure data directory exists
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://github.com/BKirkby/csusb_spring26_cse5140_team1/"
    "releases/download/v1.0-data/train.csv"
)

if not DATA_PATH.exists():
    print("Downloading dataset …")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")
else:
    print("Dataset already exists. Skipping download.")

# Load first 1,000,000 rows (same as Phase 1 / Phase 2)
print("Loading dataset into memory …")
df = pd.read_csv(DATA_PATH, nrows=NROWS)
print("Loaded df:", df.shape)

# Shuffle identically to Phase 1
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("Shuffled with seed:", SEED)

# ---- Exact same split as Phase 1 ----
dev_df, holdout_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED)
train_df, val_df   = train_test_split(dev_df, test_size=VAL_RATIO, random_state=SEED)
test_df = holdout_df  # alias for clarity

print(f"\ntrain_df:   {train_df.shape}")
print(f"val_df:     {val_df.shape}")
print(f"holdout_df: {holdout_df.shape}")

# Quick sanity check — row counts should match Phase 1
assert len(train_df) + len(val_df) + len(holdout_df) == NROWS, "Split sizes do not sum to NROWS!"
print("\n✓ Split verification passed — row counts sum to", NROWS)


> **Cell 3 -- Feature Engineering (Same Pipeline as Phases 1 & 2)**
>
> Apply the same `build_features()` function from Phase 1, then select only the 6 fuzzy input features.


In [ ]:

# ============================================================
# Cell 3 — Feature Engineering (Same Pipeline as Phase 1)
# Then select only the 6 fuzzy input features.
# ============================================================

def haversine_km_fn(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in km."""
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def build_features(dfin: pd.DataFrame) -> pd.DataFrame:
    """Exact replica of Phase 1 build_features()."""
    X = pd.DataFrame(index=dfin.index)

    # Temporal
    dt = pd.to_datetime(dfin["pickup_datetime"], errors="coerce")
    X["pickup_hour"]  = dt.dt.hour.fillna(0).astype(int)
    X["pickup_dow"]   = dt.dt.dayofweek.fillna(0).astype(int)
    X["pickup_month"] = dt.dt.month.fillna(0).astype(int)

    X["hour_sin"] = np.sin(2 * np.pi * X["pickup_hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["pickup_hour"] / 24)
    X["dow_sin"]  = np.sin(2 * np.pi * X["pickup_dow"] / 7)
    X["dow_cos"]  = np.cos(2 * np.pi * X["pickup_dow"] / 7)

    # Spatial
    X["delta_lat"] = (dfin["dropoff_latitude"] - dfin["pickup_latitude"]).astype(float)
    X["delta_lon"] = (dfin["dropoff_longitude"] - dfin["pickup_longitude"]).astype(float)
    X["haversine_km"] = haversine_km_fn(
        dfin["pickup_latitude"].astype(float),
        dfin["pickup_longitude"].astype(float),
        dfin["dropoff_latitude"].astype(float),
        dfin["dropoff_longitude"].astype(float),
    )

    # Proxies
    X["passenger_count"] = pd.to_numeric(dfin["passenger_count"], errors="coerce").fillna(0.0)
    X["store_and_fwd_Y"] = (dfin["store_and_fwd_flag"].astype(str).str.upper() == "Y").astype(int)

    vendor_oh = pd.get_dummies(dfin["vendor_id"].astype(str), prefix="vendor", drop_first=False)
    X = pd.concat([X, vendor_oh], axis=1)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X


# ---- Build features for train and validation ----
X_train_full = build_features(train_df)
X_val_full   = build_features(val_df).reindex(columns=X_train_full.columns, fill_value=0.0)

y_train = train_df[TARGET].to_numpy().astype(np.float64)
y_val   = val_df[TARGET].to_numpy().astype(np.float64)

# ---- Select only the 6 fuzzy input features ----
X_train_fs = X_train_full[FUZZY_FEATURES].copy()
X_val_fs   = X_val_full[FUZZY_FEATURES].copy()

print("Full feature set:", list(X_train_full.columns))
print(f"\nFuzzy subset: {FUZZY_FEATURES}")
print(f"X_train_fs: {X_train_fs.shape}   X_val_fs: {X_val_fs.shape}")
print(f"y_train:    {y_train.shape}       y_val:   {y_val.shape}")

# Quick distribution check for fuzzy universe clipping
print("\n--- Fuzzy feature summary (train) ---")
print(X_train_fs.describe().T[["min", "25%", "50%", "75%", "max"]].to_string())


> **Cell 4 -- Define Membership Functions**
>
> Instantiates the six fuzzy inputs and one fuzzy output exactly as locked in Section 1.5.


In [4]:
# Step 2 — implementation-ready membership function specification
# Run this after imports and after setting up skfuzzy / ctrl imports in Cell 1.

import numpy as np
import pandas as pd
import skfuzzy as fuzz
import skfuzzy.control as ctrl

# -----------------------------
# 1) Universes
# -----------------------------
haversine_km = ctrl.Antecedent(np.arange(0, 30.01, 0.1), 'haversine_km')
pickup_hour = ctrl.Antecedent(np.arange(0, 24, 1), 'pickup_hour')
pickup_dow = ctrl.Antecedent(np.arange(0, 7, 1), 'pickup_dow')
delta_lat = ctrl.Antecedent(np.arange(-0.20, 0.201, 0.001), 'delta_lat')
delta_lon = ctrl.Antecedent(np.arange(-0.20, 0.201, 0.001), 'delta_lon')
passenger_count = ctrl.Antecedent(np.arange(1, 6.01, 0.1), 'passenger_count')
predicted_duration = ctrl.Consequent(np.arange(0, 5401, 1), 'predicted_duration')

# -----------------------------
# 2) Exact membership specifications
# -----------------------------
membership_specs = {
    'haversine_km': {
        'family': 'Spatial',
        'range': '0 to 30 km',
        'labels': ['short', 'medium', 'long'],
        'membership_type': ['trap', 'tri', 'trap'],
        'exact_parameters': [[0, 0, 1.5, 4.0], [2.5, 7.5, 12.5], [10.0, 16.0, 30.0, 30.0]],
        'role_in_rules': 'Core baseline driver'
    },
    'pickup_hour': {
        'family': 'Temporal',
        'range': '0 to 23',
        'labels': ['early_morning', 'morning_rush', 'midday', 'evening_rush', 'night'],
        'membership_type': ['trap', 'tri', 'tri', 'tri', 'trap'],
        'exact_parameters': [[0, 0, 5, 7], [6, 8, 10], [10, 13, 16], [15, 18, 20], [19, 21, 23, 23]],
        'role_in_rules': 'Traffic-pattern modifier'
    },
    'pickup_dow': {
        'family': 'Temporal',
        'range': '0 to 6',
        'labels': ['weekday', 'weekend'],
        'membership_type': ['trap', 'trap'],
        'exact_parameters': [[0, 0, 3.5, 4.5], [4.5, 5.0, 6.0, 6.0]],
        'role_in_rules': 'Day-type modifier'
    },
    'delta_lat': {
        'family': 'Spatial',
        'range': '-0.20 to 0.20',
        'labels': ['south', 'neutral', 'north'],
        'membership_type': ['tri', 'tri', 'tri'],
        'exact_parameters': [[-0.20, -0.10, -0.01], [-0.03, 0.00, 0.03], [0.01, 0.10, 0.20]],
        'role_in_rules': 'North/south directional modifier'
    },
    'delta_lon': {
        'family': 'Spatial',
        'range': '-0.20 to 0.20',
        'labels': ['west', 'neutral', 'east'],
        'membership_type': ['tri', 'tri', 'tri'],
        'exact_parameters': [[-0.20, -0.10, -0.01], [-0.03, 0.00, 0.03], [0.01, 0.10, 0.20]],
        'role_in_rules': 'East/west directional modifier'
    },
    'passenger_count': {
        'family': 'Context',
        'range': '1 to 6',
        'labels': ['low', 'medium', 'high'],
        'membership_type': ['trap', 'tri', 'trap'],
        'exact_parameters': [[1.0, 1.0, 1.5, 2.0], [1.5, 3.0, 4.5], [4.0, 5.0, 6.0, 6.0]],
        'role_in_rules': 'Sparse tie-breaker context variable'
    },
    'predicted_duration': {
        'family': 'Output',
        'range': '0 to 5400 sec',
        'labels': ['very_short', 'short', 'medium', 'long', 'very_long'],
        'membership_type': ['trap', 'tri', 'tri', 'tri', 'trap'],
        'exact_parameters': [[0, 0, 180, 420], [240, 600, 960], [780, 1500, 2280], [1800, 3000, 4200], [3600, 4500, 5400, 5400]],
        'role_in_rules': 'Consequent space'
    }
}

# -----------------------------
# 3) Input membership functions
# -----------------------------
# haversine_km
haversine_km['short'] = fuzz.trapmf(haversine_km.universe, [0, 0, 1.5, 4.0])
haversine_km['medium'] = fuzz.trimf(haversine_km.universe, [2.5, 7.5, 12.5])
haversine_km['long'] = fuzz.trapmf(haversine_km.universe, [10.0, 16.0, 30.0, 30.0])

# pickup_hour
pickup_hour['early_morning'] = fuzz.trapmf(pickup_hour.universe, [0, 0, 5, 7])
pickup_hour['morning_rush'] = fuzz.trimf(pickup_hour.universe, [6, 8, 10])
pickup_hour['midday'] = fuzz.trimf(pickup_hour.universe, [10, 13, 16])
pickup_hour['evening_rush'] = fuzz.trimf(pickup_hour.universe, [15, 18, 20])
pickup_hour['night'] = fuzz.trapmf(pickup_hour.universe, [19, 21, 23, 23])

# pickup_dow
pickup_dow['weekday'] = fuzz.trapmf(pickup_dow.universe, [0, 0, 3.5, 4.5])
pickup_dow['weekend'] = fuzz.trapmf(pickup_dow.universe, [4.5, 5.0, 6.0, 6.0])

# delta_lat
delta_lat['south'] = fuzz.trimf(delta_lat.universe, [-0.20, -0.10, -0.01])
delta_lat['neutral'] = fuzz.trimf(delta_lat.universe, [-0.03, 0.00, 0.03])
delta_lat['north'] = fuzz.trimf(delta_lat.universe, [0.01, 0.10, 0.20])

# delta_lon
delta_lon['west'] = fuzz.trimf(delta_lon.universe, [-0.20, -0.10, -0.01])
delta_lon['neutral'] = fuzz.trimf(delta_lon.universe, [-0.03, 0.00, 0.03])
delta_lon['east'] = fuzz.trimf(delta_lon.universe, [0.01, 0.10, 0.20])

# passenger_count
passenger_count['low'] = fuzz.trapmf(passenger_count.universe, [1.0, 1.0, 1.5, 2.0])
passenger_count['medium'] = fuzz.trimf(passenger_count.universe, [1.5, 3.0, 4.5])
passenger_count['high'] = fuzz.trapmf(passenger_count.universe, [4.0, 5.0, 6.0, 6.0])

# -----------------------------
# 4) Output membership functions
# -----------------------------
predicted_duration['very_short'] = fuzz.trapmf(predicted_duration.universe, [0, 0, 180, 420])
predicted_duration['short'] = fuzz.trimf(predicted_duration.universe, [240, 600, 960])
predicted_duration['medium'] = fuzz.trimf(predicted_duration.universe, [780, 1500, 2280])
predicted_duration['long'] = fuzz.trimf(predicted_duration.universe, [1800, 3000, 4200])
predicted_duration['very_long'] = fuzz.trapmf(predicted_duration.universe, [3600, 4500, 5400, 5400])

predicted_duration.defuzzify_method = 'centroid'

# -----------------------------
# 5) Exportable tables for documentation / QA
# -----------------------------
membership_parameter_table = pd.DataFrame([
    ['haversine_km', 'Spatial', '0 to 30 km', 3, 'short, medium, long', 'trap / tri / trap', '[0,0,1.5,4.0] | [2.5,7.5,12.5] | [10,16,30,30]', 'Core baseline driver'],
    ['pickup_hour', 'Temporal', '0 to 23', 5, 'early_morning, morning_rush, midday, evening_rush, night', 'trap / tri / tri / tri / trap', '[0,0,5,7] | [6,8,10] | [10,13,16] | [15,18,20] | [19,21,23,23]', 'Traffic-pattern modifier'],
    ['pickup_dow', 'Temporal', '0 to 6', 2, 'weekday, weekend', 'trap / trap', '[0,0,3.5,4.5] | [4.5,5,6,6]', 'Day-type modifier'],
    ['delta_lat', 'Spatial', '-0.20 to 0.20', 3, 'south, neutral, north', 'tri / tri / tri', '[-0.20,-0.10,-0.01] | [-0.03,0.00,0.03] | [0.01,0.10,0.20]', 'North/south directional modifier'],
    ['delta_lon', 'Spatial', '-0.20 to 0.20', 3, 'west, neutral, east', 'tri / tri / tri', '[-0.20,-0.10,-0.01] | [-0.03,0.00,0.03] | [0.01,0.10,0.20]', 'East/west directional modifier'],
    ['passenger_count', 'Context', '1 to 6', 3, 'low, medium, high', 'trap / tri / trap', '[1,1,1.5,2] | [1.5,3,4.5] | [4,5,6,6]', 'Sparse tie-breaker context variable'],
    ['predicted_duration', 'Output', '0 to 5400 sec', 5, 'very_short, short, medium, long, very_long', 'trap / tri / tri / tri / trap', '[0,0,180,420] | [240,600,960] | [780,1500,2280] | [1800,3000,4200] | [3600,4500,5400,5400]', 'Consequent space'],
], columns=['variable', 'family', 'range', 'n_sets', 'labels', 'membership_type', 'exact_parameters', 'role_in_rules'])

membership_range_table = membership_parameter_table.copy()


> **Cell 4b -- Membership Function Visualizations**
>
> Plots all 7 variables' membership functions side-by-side for visual verification against Section 1.5 specs, plus a feature importance bar chart and rule family distribution.


In [ ]:

# ============================================================
# Cell 4b — Membership Function Plots + Feature Importance + Rule Family Chart
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Color palette ---
COLORS = ['#2e4a6e', '#e8850e', '#2d8a4e', '#a33', '#6c5ce7', '#00b894', '#fdcb6e']

# ========================================
# FIGURE 1: All Membership Functions (4x2)
# ========================================
variables = [
    ('haversine_km', haversine_km, 'Haversine Distance (km)'),
    ('pickup_hour', pickup_hour, 'Pickup Hour'),
    ('pickup_dow', pickup_dow, 'Pickup Day of Week'),
    ('delta_lat', delta_lat, 'Delta Latitude'),
    ('delta_lon', delta_lon, 'Delta Longitude'),
    ('passenger_count', passenger_count, 'Passenger Count'),
    ('predicted_duration', predicted_duration, 'Predicted Duration (sec)'),
]

fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()

for idx, (name, var, title) in enumerate(variables):
    ax = axes[idx]
    for i, label in enumerate(var.terms):
        ax.plot(var.universe, var[label].mf, linewidth=2.2, color=COLORS[i % len(COLORS)], label=label)
        ax.fill_between(var.universe, var[label].mf, alpha=0.12, color=COLORS[i % len(COLORS)])
    ax.set_title(title, fontsize=13, fontweight='bold', color='#0d1b2a', pad=10)
    ax.set_ylabel('Membership', fontsize=10, color='#333')
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide the empty 8th subplot
axes[7].set_visible(False)

fig.suptitle('Membership Functions — All Fuzzy Variables', fontsize=16, fontweight='bold', color='#0d1b2a', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ========================================
# FIGURE 2: Feature Importance Bar Chart
# ========================================
fig2, ax2 = plt.subplots(figsize=(10, 5))
features = ['haversine_km', 'hour_cos', 'delta_lat', 'pickup_hour', 'dow_cos', 'delta_lon', 'pickup_dow', 'passenger_count']
importance = [2.0482, 0.3014, 0.2860, 0.1077, 0.0846, 0.0644, 0.0477, 0.0038]
selected = [True, False, True, True, False, True, True, True]  # which ones we selected
bar_colors = ['#2e4a6e' if s else '#c0c8d0' for s in selected]

bars = ax2.barh(features[::-1], importance[::-1], color=bar_colors[::-1], edgecolor='white', height=0.65)
ax2.set_xlabel('R² Drop (Permutation Importance)', fontsize=11, color='#333')
ax2.set_title('Phase 1 NN Feature Importance — Selected vs. Excluded for FS', fontsize=13, fontweight='bold', color='#0d1b2a')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='x', alpha=0.25, linestyle='--')

legend_elements = [mpatches.Patch(facecolor='#2e4a6e', label='Selected for FS'),
                   mpatches.Patch(facecolor='#c0c8d0', label='Excluded')]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

# ========================================
# FIGURE 3: Rule Family Distribution
# ========================================
fig3, ax3 = plt.subplots(figsize=(10, 5))
families = ['Distance\nbaseline', 'Distance +\nrush-hour', 'Distance +\nnight/early', 'Distance +\nmidday', 'Distance +\nday type', 'Distance +\ndirection', 'Context\nexceptions']
counts = [3, 6, 6, 3, 4, 6, 2]
family_colors = ['#0d1b2a', '#a33', '#2e4a6e', '#e8850e', '#6c5ce7', '#2d8a4e', '#6c757d']

bars3 = ax3.bar(families, counts, color=family_colors, edgecolor='white', width=0.7)
ax3.set_ylabel('Number of Rules', fontsize=11, color='#333')
ax3.set_title('Rule Distribution by Family (30 total rules)', fontsize=13, fontweight='bold', color='#0d1b2a')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.grid(axis='y', alpha=0.25, linestyle='--')

for bar, count in zip(bars3, counts):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15, str(count),
             ha='center', va='bottom', fontweight='bold', fontsize=11, color='#0d1b2a')

ax3.axhline(y=25, color='#2d8a4e', linestyle='--', alpha=0.5, label='Budget min (25)')
ax3.axhline(y=40, color='#a33', linestyle='--', alpha=0.5, label='Budget max (40)')
plt.tight_layout()
plt.show()

# ========================================
# FIGURE 4: Grouped Importance Donut Chart
# ========================================
fig4, ax4 = plt.subplots(figsize=(6, 6))
family_names = ['Spatial', 'Temporal', 'Context']
family_drops = [2.399, 0.639, 0.123]
family_cols = ['#2e4a6e', '#e8850e', '#6c757d']

wedges, texts, autotexts = ax4.pie(family_drops, labels=family_names, colors=family_cols,
                                     autopct='%1.1f%%', startangle=90, pctdistance=0.78,
                                     textprops={'fontsize': 12, 'color': '#0d1b2a'})
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
    at.set_color('white')

centre_circle = plt.Circle((0,0), 0.55, fc='white')
ax4.add_artist(centre_circle)
ax4.set_title('Feature Importance by Family', fontsize=13, fontweight='bold', color='#0d1b2a', pad=15)
plt.tight_layout()
plt.show()

print("All visualizations rendered.")


> **Cell 5 -- Define Fuzzy Rule Base (30 Rules)**
>
> Translates the Section 1.7 rule families into 30 `ctrl.Rule(...)` objects, within the locked complexity caps from Section 1.6.


In [ ]:

# ============================================================
# Cell 5 — Full Fuzzy Rule Base
# Aligned with Section 1.7 rule families and Section 1.6 budget.
# Target: 25–40 rules, avg 2–3 antecedents, max 4.
# ============================================================

# ---- Interpretability budget (from Section 1.6) ----
rule_budget = {
    "target_active_rule_count": (25, 40),
    "hard_rule_ceiling": 50,
    "target_average_antecedents": (2, 3),
    "hard_antecedent_ceiling": 4,
}

# ===========================================================
# Family 1: Distance baseline (haversine_km only)
# ===========================================================
r01 = ctrl.Rule(haversine_km['short'],  predicted_duration['very_short'], label='r01_dist_short')
r02 = ctrl.Rule(haversine_km['medium'], predicted_duration['medium'],     label='r02_dist_medium')
r03 = ctrl.Rule(haversine_km['long'],   predicted_duration['long'],       label='r03_dist_long')

# ===========================================================
# Family 2: Distance + rush-hour traffic
# ===========================================================
r04 = ctrl.Rule(haversine_km['short']  & pickup_hour['morning_rush'], predicted_duration['short'],     label='r04_short_morning_rush')
r05 = ctrl.Rule(haversine_km['medium'] & pickup_hour['morning_rush'], predicted_duration['long'],      label='r05_medium_morning_rush')
r06 = ctrl.Rule(haversine_km['long']   & pickup_hour['morning_rush'], predicted_duration['very_long'], label='r06_long_morning_rush')
r07 = ctrl.Rule(haversine_km['short']  & pickup_hour['evening_rush'], predicted_duration['short'],     label='r07_short_evening_rush')
r08 = ctrl.Rule(haversine_km['medium'] & pickup_hour['evening_rush'], predicted_duration['long'],      label='r08_medium_evening_rush')
r09 = ctrl.Rule(haversine_km['long']   & pickup_hour['evening_rush'], predicted_duration['very_long'], label='r09_long_evening_rush')

# ===========================================================
# Family 3: Distance + late-night / early-morning relief
# ===========================================================
r10 = ctrl.Rule(haversine_km['short']  & pickup_hour['night'],         predicted_duration['very_short'], label='r10_short_night')
r11 = ctrl.Rule(haversine_km['medium'] & pickup_hour['night'],         predicted_duration['short'],      label='r11_medium_night')
r12 = ctrl.Rule(haversine_km['long']   & pickup_hour['night'],         predicted_duration['medium'],     label='r12_long_night')
r13 = ctrl.Rule(haversine_km['short']  & pickup_hour['early_morning'], predicted_duration['very_short'], label='r13_short_early')
r14 = ctrl.Rule(haversine_km['medium'] & pickup_hour['early_morning'], predicted_duration['short'],      label='r14_medium_early')
r15 = ctrl.Rule(haversine_km['long']   & pickup_hour['early_morning'], predicted_duration['medium'],     label='r15_long_early')

# ===========================================================
# Family 4: Distance + midday (moderate traffic)
# ===========================================================
r16 = ctrl.Rule(haversine_km['short']  & pickup_hour['midday'], predicted_duration['very_short'], label='r16_short_midday')
r17 = ctrl.Rule(haversine_km['medium'] & pickup_hour['midday'], predicted_duration['medium'],     label='r17_medium_midday')
r18 = ctrl.Rule(haversine_km['long']   & pickup_hour['midday'], predicted_duration['long'],       label='r18_long_midday')

# ===========================================================
# Family 5: Distance + day type (weekday / weekend)
# ===========================================================
r19 = ctrl.Rule(haversine_km['long']   & pickup_dow['weekday'], predicted_duration['very_long'], label='r19_long_weekday')
r20 = ctrl.Rule(haversine_km['medium'] & pickup_dow['weekday'], predicted_duration['medium'],    label='r20_medium_weekday')
r21 = ctrl.Rule(haversine_km['medium'] & pickup_dow['weekend'], predicted_duration['short'],     label='r21_medium_weekend')
r22 = ctrl.Rule(haversine_km['long']   & pickup_dow['weekend'], predicted_duration['long'],      label='r22_long_weekend')

# ===========================================================
# Family 6: Distance + directional displacement
# ===========================================================
r23 = ctrl.Rule(haversine_km['medium'] & delta_lat['north'], predicted_duration['long'],   label='r23_medium_north')
r24 = ctrl.Rule(haversine_km['medium'] & delta_lat['south'], predicted_duration['long'],   label='r24_medium_south')
r25 = ctrl.Rule(haversine_km['medium'] & delta_lon['east'],  predicted_duration['long'],   label='r25_medium_east')
r26 = ctrl.Rule(haversine_km['medium'] & delta_lon['west'],  predicted_duration['long'],   label='r26_medium_west')
r27 = ctrl.Rule(haversine_km['long']   & delta_lat['north'], predicted_duration['very_long'], label='r27_long_north')
r28 = ctrl.Rule(haversine_km['long']   & delta_lat['south'], predicted_duration['very_long'], label='r28_long_south')

# ===========================================================
# Family 7: Context (passenger_count) — sparse tie-breakers
# ===========================================================
r29 = ctrl.Rule(haversine_km['medium'] & passenger_count['high'], predicted_duration['long'],  label='r29_medium_high_pax')
r30 = ctrl.Rule(haversine_km['short']  & passenger_count['low'],  predicted_duration['very_short'], label='r30_short_low_pax')

# ===========================================================
# Collect all rules
# ===========================================================
all_rules = [
    r01, r02, r03,
    r04, r05, r06, r07, r08, r09,
    r10, r11, r12, r13, r14, r15,
    r16, r17, r18,
    r19, r20, r21, r22,
    r23, r24, r25, r26, r27, r28,
    r29, r30,
]

# ---- Interpretability metrics ----
n_rules = len(all_rules)
antecedent_lengths = []
for r in all_rules:
    # Count antecedent terms by counting '&' in the rule's string repr + 1
    rule_str = str(r)
    n_terms = rule_str.count("AND") + rule_str.count("&") + 1
    # More reliable: count the number of fuzzy variable references
    count = sum(1 for v in ['haversine_km', 'pickup_hour', 'pickup_dow',
                            'delta_lat', 'delta_lon', 'passenger_count']
                if v in str(r.antecedent))
    antecedent_lengths.append(max(count, 1))

avg_length = np.mean(antecedent_lengths)
max_length = max(antecedent_lengths)

print(f"Total active rules:       {n_rules}")
print(f"Average antecedent length: {avg_length:.1f}")
print(f"Max antecedent length:     {max_length}")
print(f"Within budget [25-40]:     {'YES' if 25 <= n_rules <= 40 else 'NO'}")
print(f"Avg within [2-3]:          {'YES' if 2 <= avg_length <= 3 else 'CLOSE' if avg_length < 2 else 'NO'}")
print(f"Max ≤ 4:                   {'YES' if max_length <= 4 else 'NO'}")

# Show rule family summary
print("\n--- Rule summary ---")
for i, r in enumerate(all_rules, 1):
    print(f"  {r.label}")


> **Cell 6 -- Build & Run Fuzzy Inference System**
>
> Create the control system, run predictions on **validation data only**, and compute RMSE, MAE, R-squared, and MAPE. Test set remains untouched.


In [ ]:

# ============================================================
# Cell 6 — Build & Run Fuzzy Inference System
# Predicts on VALIDATION set only.  Test set remains untouched.
# ============================================================
import time

# ---- 1) Build the ControlSystem ----
fs_ctrl = ctrl.ControlSystem(all_rules)
fs_sim  = ctrl.ControlSystemSimulation(fs_ctrl)
print(f"ControlSystem built with {len(all_rules)} rules.")

# ---- 2) Clip helper — keep inputs within fuzzy universes ----
CLIP_BOUNDS = {
    "haversine_km":    (0.0, 30.0),
    "pickup_hour":     (0, 23),
    "pickup_dow":      (0, 6),
    "delta_lat":       (-0.20, 0.20),
    "delta_lon":       (-0.20, 0.20),
    "passenger_count": (1.0, 6.0),
}

def clip_row(row):
    out = {}
    for feat, (lo, hi) in CLIP_BOUNDS.items():
        out[feat] = np.clip(row[feat], lo, hi)
    return out


# ---- 3) Row-level prediction with fallback ----
FALLBACK_DURATION = 600.0   # 10-min median fallback if no rules fire

def predict_one(sim, row_dict):
    """Feed one observation into the FIS and return defuzzified output."""
    for feat, val in row_dict.items():
        sim.input[feat] = val
    try:
        sim.compute()
        return sim.output["predicted_duration"]
    except Exception:
        return FALLBACK_DURATION


# ---- 4) Run on validation set ----
print(f"\nRunning FIS on {len(X_val_fs):,} validation samples …")
t0 = time.time()

val_preds = np.full(len(X_val_fs), np.nan)
n_fallback = 0

for i, (idx, row) in enumerate(X_val_fs.iterrows()):
    clipped = clip_row(row)
    pred = predict_one(fs_sim, clipped)
    if pred == FALLBACK_DURATION:
        n_fallback += 1
    val_preds[i] = pred

    if (i + 1) % 25_000 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(X_val_fs) - i - 1) / rate
        print(f"  [{i+1:>7,} / {len(X_val_fs):,}]  "
              f"{rate:,.0f} rows/s  ETA {eta:.0f}s  fallbacks so far: {n_fallback}")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s  ({len(X_val_fs)/elapsed:,.0f} rows/s)")
print(f"Fallback predictions: {n_fallback:,} / {len(X_val_fs):,} "
      f"({100*n_fallback/len(X_val_fs):.2f}%)")

# ---- 5) Validation metrics (test set NOT used) ----
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
mae  = mean_absolute_error(y_val, val_preds)
r2   = r2_score(y_val, val_preds)

# MAPE — exclude zeros in target to avoid division issues
mask = y_val > 0
mape = np.mean(np.abs((y_val[mask] - val_preds[mask]) / y_val[mask])) * 100

print("\n========== VALIDATION METRICS (test set untouched) ==========")
print(f"  RMSE : {rmse:,.2f} seconds")
print(f"  MAE  : {mae:,.2f} seconds")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
print("=============================================================")

# ---- 6) Save validation predictions for later comparison ----
val_results = pd.DataFrame({
    "y_true": y_val,
    "y_pred": val_preds,
    "residual": y_val - val_preds,
})
print(f"\nPrediction distribution:\n{val_results['y_pred'].describe().to_string()}")


> **Cell 7 -- Validation Results Visualization**
>
> Predicted vs. actual scatter, residual distribution, and prediction distribution histogram for validation set results.


In [ ]:

# ============================================================
# Cell 7 — Validation Results Visualization
# (Only uses validation data — test set remains untouched)
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# --- 1) Predicted vs Actual scatter ---
ax1 = axes[0, 0]
sample_idx = np.random.choice(len(y_val), size=min(15000, len(y_val)), replace=False)
ax1.scatter(y_val[sample_idx], val_preds[sample_idx], alpha=0.08, s=6, color='#2e4a6e')
lim = max(y_val[sample_idx].max(), val_preds[sample_idx].max())
ax1.plot([0, lim], [0, lim], '--', color='#a33', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('Actual Duration (sec)', fontsize=11, color='#333')
ax1.set_ylabel('Predicted Duration (sec)', fontsize=11, color='#333')
ax1.set_title('Predicted vs. Actual (Validation Set)', fontsize=13, fontweight='bold', color='#0d1b2a')
ax1.set_xlim(0, 6000)
ax1.set_ylim(0, 6000)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.2, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- 2) Residual Distribution ---
ax2 = axes[0, 1]
residuals = y_val - val_preds
ax2.hist(residuals, bins=120, range=(-3000, 3000), color='#2e4a6e', edgecolor='white', alpha=0.85)
ax2.axvline(x=0, color='#a33', linestyle='--', linewidth=1.5)
ax2.axvline(x=np.median(residuals), color='#e8850e', linestyle='-', linewidth=1.5, label=f'Median = {np.median(residuals):.0f}s')
ax2.set_xlabel('Residual (Actual - Predicted) [sec]', fontsize=11, color='#333')
ax2.set_ylabel('Count', fontsize=11, color='#333')
ax2.set_title('Residual Distribution', fontsize=13, fontweight='bold', color='#0d1b2a')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.2, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# --- 3) Prediction Distribution vs Actual ---
ax3 = axes[1, 0]
ax3.hist(y_val, bins=100, range=(0, 5400), color='#2e4a6e', alpha=0.5, label='Actual', edgecolor='white')
ax3.hist(val_preds, bins=100, range=(0, 5400), color='#e8850e', alpha=0.5, label='Predicted', edgecolor='white')
ax3.set_xlabel('Duration (sec)', fontsize=11, color='#333')
ax3.set_ylabel('Count', fontsize=11, color='#333')
ax3.set_title('Actual vs. Predicted Duration Distribution', fontsize=13, fontweight='bold', color='#0d1b2a')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.2, linestyle='--')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# --- 4) Validation Metrics Summary Card ---
ax4 = axes[1, 1]
ax4.axis('off')
metrics_text = (
    f"VALIDATION METRICS\n"
    f"(test set untouched)\n\n"
    f"RMSE:  {rmse:,.0f} sec\n"
    f"MAE:   {mae:,.0f} sec\n"
    f"R²:    {r2:.4f}\n"
    f"MAPE:  {mape:.1f}%\n\n"
    f"Rules:         {n_rules}\n"
    f"Avg length:    {avg_length:.1f}\n"
    f"Max length:    {max_length}\n"
    f"Fallbacks:     {n_fallback:,}"
)
ax4.text(0.5, 0.5, metrics_text, transform=ax4.transAxes, fontsize=14,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace', color='#0d1b2a',
         bbox=dict(boxstyle='round,pad=1', facecolor='#f0f4f8', edgecolor='#2e4a6e', linewidth=2))

fig.suptitle('Step 2 — Fuzzy System Validation Results', fontsize=16, fontweight='bold', color='#0d1b2a', y=1.01)
plt.tight_layout()
plt.show()

print("Validation visualizations complete. Test set remains untouched.")
